# 🚗 Preparación de Datos y Entrenamiento de YOLOv8 en Kaggle

Este notebook contiene el código necesario para:
1. **Convertir anotaciones COCO JSON** a formato **YOLO TXT**.
2. **Crear el archivo `data.yaml`** requerido por YOLOv8.
3. **Ejecutar el entrenamiento (Fine-Tuning)** a partir de tus pesos previos `best.pt`.

---

## 🛠️ Paso 1: Definir Función de Conversión (COCO ➔ YOLO)

In [ ]:
import json
import os
import shutil
from pathlib import Path

def convertir_coco_a_yolo(json_path, images_dir, output_dir, class_mapping):
    """
    Convierte anotaciones COCO JSON a archivos de etiquetas individuales YOLO TXT.
    """
    os.makedirs(os.path.join(output_dir, "images"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "labels"), exist_ok=True)
    
    if not os.path.exists(json_path):
        print(f"⚠️ Archivo JSON no encontrado: {json_path}")
        return
        
    print(f"🔄 Cargando anotaciones desde {json_path}...")
    with open(json_path, 'r') as f:
        data = json.load(f)
        
    images = {img['id']: img for img in data['images']}
    annotations_by_image = {}
    for ann in data['annotations']:
        img_id = ann['image_id']
        annotations_by_image.setdefault(img_id, []).append(ann)
        
    print(f"✏️ Procesando {len(images)} imágenes...")
    convertidas = 0
    
    for img_id, anns in annotations_by_image.items():
        if img_id not in images:
            continue
        img_info = images[img_id]
        filename = img_info['file_name']
        img_w = img_info['width']
        img_h = img_info['height']
        
        src_image_path = os.path.join(images_dir, filename)
        dst_image_path = os.path.join(output_dir, "images", filename)
        
        if not os.path.exists(src_image_path):
            continue
            
        shutil.copy(src_image_path, dst_image_path)
        
        # Crear archivo .txt de etiquetas
        txt_filename = Path(filename).with_suffix('.txt')
        txt_path = os.path.join(output_dir, "labels", txt_filename)
        
        with open(txt_path, 'w') as out_f:
            for ann in anns:
                coco_cat_id = ann['category_id']
                if coco_cat_id not in class_mapping:
                    continue
                yolo_class_id = class_mapping[coco_cat_id]
                
                bbox = ann['bbox'] # [x_min, y_min, width, height]
                x_center = (bbox[0] + bbox[2] / 2.0) / img_w
                y_center = (bbox[1] + bbox[3] / 2.0) / img_h
                w = bbox[2] / img_w
                h = bbox[3] / img_h
                
                out_f.write(f"{yolo_class_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n")
        convertidas += 1
        
    print(f"✅ Conversión finalizada. {convertidas} imágenes preparadas en {output_dir}")

## 🔄 Paso 2: Convertir los Datasets Vinculados

Ajusta los mapeos de clases según tus necesidades específicas para unificar en 14 clases.

In [ ]:
# Ejemplo para VehiDE Dataset
# Mapea clases de VehiDE a tus IDs de YOLO (0: scratch, 1: dent, etc.)
mapeo_vehide = {
    1: 0, # scratch
    2: 1, # dent
    # Agrega otros mapeos si es necesario
}

# Descomenta y ejecuta ajustando las rutas de tus datasets vinculados:
# convertir_coco_a_yolo(
#     json_path='/kaggle/input/vehicle-dataset-automatic-vehicle-damage-detection/0Train_via_annos.json',
#     images_dir='/kaggle/input/vehicle-dataset-automatic-vehicle-damage-detection/Otrain',
#     output_dir='/kaggle/working/dataset_final/train',
#     class_mapping=mapeo_vehide
# )

## 📄 Paso 3: Generar `data.yaml`

In [ ]:
yaml_content = """
path: /kaggle/working/dataset_final
train: images/train
val: images/val

names:
  0: scratch
  1: dent
  2: crack
  3: glass_crack
  4: headlight_damage
  5: taillight_damage
  6: mirror_damage
  7: bumper_damage
  8: door_damage
  9: fender_damage
  10: hood_damage
  11: trunk_damage
  12: roof_damage
  13: rust
"""

with open('/kaggle/working/data.yaml', 'w') as f:
    f.write(yaml_content.strip())
print("✅ Archivo data.yaml generado con éxito.")

## 🏋️ Paso 4: Instalar Librería e Iniciar Entrenamiento

In [ ]:
!pip install ultralytics -q

In [ ]:
from ultralytics import YOLO

# Cargar tu archivo de pesos actual (cárgalo previamente en 'Input' desde tu PC)
# Si deseas arrancar de cero usa YOLO('yolov8s.pt')
ruta_pesos_iniciales = '/kaggle/input/tus-pesos-previos/best.pt'

if os.path.exists(ruta_pesos_iniciales):
    model = YOLO(ruta_pesos_iniciales)
    print("🔥 Cargando pesos entrenados previos para continuar...")
else:
    model = YOLO('yolov8s.pt')
    print("🌱 Pesos previos no encontrados. Arrancando con YOLOv8s base...")

# Ejecutar entrenamiento (Fine-Tuning)
model.train(
    data='/kaggle/working/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project='inspeccion',
    name='yolov8_danos_mejorado'
)